# Jalon M3 : transformation et statistiques
## 1. Variables temporelles en heure de Paris
Je recharge les données propres de M2, puis je crée des variables (heure, jour, week-end, mois) à partir de `t_paris`, parce que les heures de pointe se lisent en heure locale.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("data/processed/comptages_propres.parquet")
print(df.shape)
df.dtypes

(97152, 12)


iu_ac                                string
t_paris        datetime64[us, Europe/Paris]
libelle                            category
q                                   float64
k                                   float64
etat_trafic                        category
etat_barre                         category
groupe_q                              int64
q_impute                            float64
k_impute                            float64
q_comble                               bool
k_comble                               bool
dtype: object

In [2]:
df["heure"] = df["t_paris"].dt.hour
df["jour_semaine"] = df["t_paris"].dt.dayofweek   # 0 = lundi, 6 = dimanche
df["est_weekend"] = df["jour_semaine"] >= 5
df["mois"] = df["t_paris"].dt.month
df[["t_paris", "heure", "jour_semaine", "est_weekend", "mois"]].head(3)

,t_paris,heure,jour_semaine,est_weekend,mois
0,2026-03-01 01:00:00+01:00,1,6,True,3
1,2026-03-01 02:00:00+01:00,2,6,True,3
2,2026-03-01 03:00:00+01:00,3,6,True,3


In [3]:
print(df.loc[df["t_paris"].dt.date == pd.Timestamp("2026-03-29").date(), "heure"].sort_values().unique())

[ 0  1  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


Les variables `heure`, `jour_semaine` (0 = lundi), `est_weekend` et `mois` sont créées à partir de `t_paris`. La première ligne (1er mars, 01h00) est bien un dimanche, et le 29 mars la liste des heures saute de 1 à 3.
Donc les variables sont calculées en heure de Paris, y compris autour du changement d'heure.On peut faire l'encodage maintenant.

## 2. Encodage
Avant de choisir ordinal ou one-hot, je vérifie si `etat_trafic` a un ordre. S'il en a un, l'occupation `k` doit augmenter d'un état au suivant.

In [4]:
df.groupby("etat_trafic", observed=True)["k"].agg(["median", "count"]).round(1)

,median,count
etat_trafic,,
Bloqué,58.9,1847
Fluide,5.1,72212
Inconnu,NaN,0
Pré-saturé,19.6,8635
Saturé,38.1,4097


In [5]:
ORDRE = {"Fluide": 0, "Pré-saturé": 1, "Saturé": 2, "Bloqué": 3}   # Inconnu -> NaN
df["congestion"] = df["etat_trafic"].map(ORDRE).astype("float")
df["congestion"].value_counts(dropna=False).sort_index()

congestion
0.0    72212
1.0     8635
2.0     4097
3.0     1847
NaN    10361
Name: count, dtype: int64

In [6]:
df = pd.get_dummies(df, columns=["etat_barre"], prefix="barre", dtype=int)
[c for c in df.columns if c.startswith("barre")]

['barre_Barré', 'barre_Invalide', 'barre_Ouvert']

In [7]:
(df[["barre_Barré", "barre_Invalide", "barre_Ouvert"]].sum(axis=1) == 0).sum()

np.int64(1192)

**Résultat** : l'occupation médiane `k` augmente d'un état au suivant : 5,1 (Fluide), 19,6 (Pré-saturé), 38,1 (Saturé), 58,9 (Bloqué). `Inconnu` a 0 mesure de `k`.
**Choix** :  `Inconnu` devient `NaN` ,c'est une valeur manquante.  `etat_barre` (Ouvert, Barré, Invalide) n'a pas d'ordre naturel, donc je l'encode en **one-hot** : 3 colonnes `barre_Ouvert`, `barre_Barré` et `barre_Invalide`. Les 1 192 lignes ajoutées par la grille ont 0 dans les trois colonnes, car leur état est vide.